# Viz — Matplotlib (raster + vector)

High-DPI PNG and vector exports using a shared disk projection from ``dmercator_io``.

**Data + export helper**

Loads `cache/merged.parquet` and computes stereo north `(u,v)` via `dmercator_io` so figures stay comparable with other `viz_*` notebooks. `save_hires` centralizes resolution (`dpi` for PNG, default 800) and vector formats for publication.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 200  # default 100; sharper inline figures
import numpy as np

import dmercator_io as dm

CACHE = Path("cache") / "merged.parquet"
merged = dm.load_merged_parquet(CACHE)
u, v = dm.stereo_disk_north(merged)


def save_hires(fig: plt.Figure, stem: str, dpi: int = 800) -> None:
    out = Path("cache") / "figures"
    out.mkdir(parents=True, exist_ok=True)
    fig.savefig(out / f"{stem}.png", dpi=dpi, bbox_inches="tight")
    fig.savefig(out / f"{stem}.svg", bbox_inches="tight")
    fig.savefig(out / f"{stem}.pdf", bbox_inches="tight")
    print("Wrote", stem, "PNG/SVG/PDF under", out.resolve())


**Raster plot and vector exports**

Clips extreme stereo tails for display, colors by `log1p(degree)`, then calls `save_hires` at **800 DPI** for PNG (plus SVG/PDF) under `cache/figures/`. **Interpretation:** warmer colors mark higher-degree proteins in this projection; compare files across stacks only when you use the same `stereo_disk_north` inputs.

In [ ]:
clip = 8.0
ix = (np.abs(u) < clip) & (np.abs(v) < clip)
fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(
    u[ix],
    v[ix],
    s=3,
    alpha=0.25,
    c=np.log1p(merged["degree"].to_numpy()[ix]),
    cmap="magma",
)
ax.set_aspect("equal")
ax.set_title("Stereo north — log1p(degree)")
plt.colorbar(sc, ax=ax, shrink=0.7)
save_hires(fig, "mpl_stereo_north_deg", dpi=800)
plt.show()
